[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/081.%20The%203D%20Container_Truck%20Loading%20Problem%20%28Bin%20Packing%29/P81-Tier-9_executed.ipynb)

---


# 81. The 3D Container/Truck Loading Problem

## Tier 9: A QAOA and QUBO Demonstration

This notebook demonstrates a hybrid quantum-classical approach to a small, discretized container-loading problem. It formulates candidate item placements as a Quadratic Unconstrained Binary Optimization (QUBO) model, simulates the Quantum Approximate Optimization Algorithm (QAOA) with a state vector, and uses a classical optimizer to tune the circuit parameters.

### Goals and workflow

The input is a container, a set of rectangular items, and a grid of candidate placement origins. The notebook builds one binary variable for each item–origin pair, constructs a QUBO cost matrix, selects an eight-variable subproblem, and runs a three-layer QAOA simulation on that subproblem. The resulting measurements are decoded into candidate placements and checked against the modeled constraints.

The workflow is: define the instance → formulate the QUBO → select the demonstration subproblem → simulate and optimize QAOA parameters → measure and decode bitstrings → inspect the candidate and the limits of the demonstration.

### Model assumptions and scope

Items are axis-aligned rectangular boxes and are not rotated. Candidate origins lie on a fixed grid. The model rewards item value, penalizes assigning multiple origins to the same item, penalizes overlaps between different items, and penalizes origins that would extend beyond the container. These are soft penalties in the objective, not hard constraints, so a measured bitstring must be checked explicitly. The model does not include weight limits, load-bearing capacity, stability, unloading order, uncertainty, or operational handling constraints.

The full discretized model has 96 binary variables. A state-vector simulation stores an amplitude for each computational basis state, so the required state size grows exponentially with the number of variables. This notebook therefore runs QAOA only on eight selected variables. Each selected variable represents one item placed at the single origin \((0,0,0)\); alternative origins are not included in this subproblem. The simulation illustrates an algorithmic workflow, not a solution to the full loading problem, a scalable quantum computation, or evidence of quantum advantage.

### Background: binary variables and the QUBO model

A binary variable takes the value 0 or 1. Here, \(x_{i,p}=1\) means item \(i\) is assigned to candidate origin \(p\), and \(x_{i,p}=0\) means it is not. A QUBO is an objective that is quadratic in binary variables. Penalty terms incorporate selected constraint violations into the objective while keeping the decision variables binary.

Let \(v_i\) denote the value of item \(i\), and let \(A\), \(B\), and \(M\) be the assignment, overlap, and out-of-bounds penalty coefficients. The model minimizes the following energy:

$$
E(x) = -\sum_{i,p} v_i x_{i,p}
 + A\sum_i\sum_{p<q}x_{i,p}x_{i,q}
 + B\sum_{(i,p)<(j,q),\ i<j} O_{i,p,j,q}x_{i,p}x_{j,q}
 + M\sum_{i,p} I_{i,p}x_{i,p}.
$$

The negative value term makes valuable selected placements favorable when minimizing energy. The assignment term penalizes choosing two different origins for the same item. In the overlap term, only pairs of different items are considered: \(O_{i,p,j,q}=1\) if their axis-aligned boxes overlap with positive volume and is 0 otherwise. The indicator \(I_{i,p}=1\) if item \(i\) at origin \(p\) extends beyond any container boundary and is 0 otherwise. The coefficients are modeling choices; penalties do not guarantee feasibility for every instance. In the code, `_is_out_of_bounds` computes \(I\), `_positions_overlap` computes \(O\), and `create_qubo_matrix` assembles the objective.

The code stores the QUBO coefficients in the diagonal and upper triangle of `Q`, with each pairwise term recorded once. For a binary vector \(x\), the implemented convention is

$$
E(x)=\sum_i Q_{ii}x_i + \sum_{i<j}Q_{ij}x_ix_j.
$$

The diagonal entries contain the item-value and out-of-bounds terms. Upper-triangular entries contain the assignment and overlap penalties. The indices in this compact matrix correspond to item–origin variables in the full model; the selected subproblem retains only the diagonal entry for the origin \((0,0,0)\) of each item and the pairwise coefficients between those variables.

### Background: QAOA and the hybrid optimization loop

QAOA starts from a uniform superposition of bitstrings, applies alternating problem-dependent phase and mixing operations, then measures the state. The phase operation encodes QUBO energy in the phases of the amplitudes. The mixer changes amplitudes among bitstrings. A classical optimizer adjusts the angles \(\gamma_l\) and \(\beta_l\) to seek a state with low expected QUBO energy. Since the energy is estimated from samples, the objective value is noisy and the best measured bitstring is not guaranteed to be globally optimal.

For \(p\) layers and \(n\) binary variables, the parameter vector is \((\gamma_1,\beta_1,\ldots,\gamma_p,\beta_p)\). Let \(H_C\) be the diagonal operator whose value on each computational basis state is that bitstring's QUBO energy. The state and the objective estimated by the classical optimizer are

$$
|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
= U_M(\beta_p)U_C(\gamma_p)\cdots U_M(\beta_1)U_C(\gamma_1)|+\rangle^{\otimes n},
$$

$$
F(\boldsymbol{\gamma},\boldsymbol{\beta})
=\langle\psi|H_C|\psi\rangle,
\qquad
U_C(\gamma)=e^{-i\gamma H_C},
\qquad
U_M(\beta)=e^{-i\beta\sum_{k=1}^{n}X_k}.
$$

Here, \(|+\rangle^{\otimes n}\) is the uniform starting state, and \(X_k\) is the Pauli-\(X\) operator on variable \(k\). In the code, `run_qaoa_circuit` constructs the state, `objective_function` estimates \(F\) using sampled measurements, and SciPy's COBYLA optimizer tunes the angles. `apply_phase_separator` evaluates the QUBO energy for each basis state and applies its phase; `apply_mixer_layer` performs the single-qubit rotations. The state vector contains \(2^n\) amplitudes. The implementation also constructs basis-state bit arrays, so its memory use grows at least proportionally to \(n2^n\); this exponential growth motivates the eight-variable demonstration.

COBYLA's evaluation count records how many times it evaluates the sampled objective. It is not a count of quantum circuit layers or measured bitstrings: each objective evaluation simulates a circuit and draws `shots` measurements. The fixed random seeds make the sequence of random draws reproducible when the notebook is run from the beginning in the same environment, but the objective remains a sampling-based estimate.

### What to examine

Inspect how placement variables and penalties become QUBO coefficients, how the simulator applies the QAOA layers, and how measured bitstrings are decoded and checked. The plots show the selected QUBO coefficients, the frequency of measured bitstrings, and the decoded candidate. The convergence plot records sampled objective estimates during COBYLA; because those estimates are noisy, individual fluctuations need not indicate a change in the underlying expected energy. These empirical results do not establish a speedup over classical methods.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.optimize import minimize

# Fixed seeds make the sampled measurements and initial parameters reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for the QAOA demonstration.")

In [ ]:
class QuantumContainerItem:
    """Rectangular item used in the discretized loading model."""

    def __init__(self, item_id, length, width, height, weight, value=1.0):
        self.item_id = item_id
        self.length = length
        self.width = width
        self.height = height
        self.weight = weight
        self.value = value
        self.volume = length * width * height
        self.position = None

    def get_binary_encoding_length(self, container_dims):
        """Return the number of grid-origin variables for this item."""
        grid_counts = [int(dimension / 0.5) for dimension in container_dims]
        return int(np.prod(grid_counts))


class QUBOFormulator:
    """Build an upper-triangular QUBO for candidate item placements."""

    def __init__(self, container_dims, items):
        self.container_dims = container_dims
        self.items = items
        self.grid_resolution = 0.5

        self.grid_x = int(container_dims[0] / self.grid_resolution)
        self.grid_y = int(container_dims[1] / self.grid_resolution)
        self.grid_z = int(container_dims[2] / self.grid_resolution)
        self.total_positions = self.grid_x * self.grid_y * self.grid_z
        self.n_variables = len(items) * self.total_positions

    def _position_coordinates(self, position_index):
        """Map a flattened grid index to its physical origin."""
        x_index = position_index % self.grid_x
        y_index = (position_index // self.grid_x) % self.grid_y
        z_index = position_index // (self.grid_x * self.grid_y)
        return (
            x_index * self.grid_resolution,
            y_index * self.grid_resolution,
            z_index * self.grid_resolution,
        )

    def _is_out_of_bounds(self, item, position_index):
        """Check whether the item's box exceeds any container boundary."""
        x, y, z = self._position_coordinates(position_index)
        return (
            x + item.length > self.container_dims[0]
            or y + item.width > self.container_dims[1]
            or z + item.height > self.container_dims[2]
        )

    def _positions_overlap(self, item1, item2, pos1, pos2):
        """Return whether two axis-aligned boxes have positive-volume overlap."""
        x1, y1, z1 = self._position_coordinates(pos1)
        x2, y2, z2 = self._position_coordinates(pos2)
        return not (
            x1 + item1.length <= x2
            or x2 + item2.length <= x1
            or y1 + item1.width <= y2
            or y2 + item2.width <= y1
            or z1 + item1.height <= z2
            or z2 + item2.height <= z1
        )

    def create_qubo_matrix(self):
        """Create the QUBO using one binary variable per item and origin."""
        Q = np.zeros((self.n_variables, self.n_variables), dtype=float)
        assignment_penalty = 10.0
        overlap_penalty = 15.0
        bounds_penalty = 15.0

        # Negative diagonal coefficients turn value maximization into minimization.
        for item_idx, item in enumerate(self.items):
            for pos_idx in range(self.total_positions):
                variable_idx = item_idx * self.total_positions + pos_idx
                Q[variable_idx, variable_idx] -= item.value
                if self._is_out_of_bounds(item, pos_idx):
                    Q[variable_idx, variable_idx] += bounds_penalty

        # A positive pairwise term discourages assigning one item to two origins.
        for item_idx in range(len(self.items)):
            start_idx = item_idx * self.total_positions
            end_idx = start_idx + self.total_positions
            for first in range(start_idx, end_idx):
                for second in range(first + 1, end_idx):
                    Q[first, second] += assignment_penalty

        # Overlapping candidate boxes cannot both be selected without a penalty.
        for item1_idx, item1 in enumerate(self.items):
            for item2_idx in range(item1_idx + 1, len(self.items)):
                item2 = self.items[item2_idx]
                for pos1 in range(self.total_positions):
                    for pos2 in range(self.total_positions):
                        if self._positions_overlap(item1, item2, pos1, pos2):
                            var1 = item1_idx * self.total_positions + pos1
                            var2 = item2_idx * self.total_positions + pos2
                            Q[min(var1, var2), max(var1, var2)] += overlap_penalty

        return Q

    def decode_solution(self, binary_solution):
        """Decode selected variables into item and origin records."""
        binary_solution = np.asarray(binary_solution, dtype=int)
        if binary_solution.size != self.n_variables:
            raise ValueError(
                f"Expected {self.n_variables} binary variables, "
                f"received {binary_solution.size}."
            )

        placements = []
        for item_idx, item in enumerate(self.items):
            for pos_idx in range(self.total_positions):
                variable_idx = item_idx * self.total_positions + pos_idx
                if binary_solution[variable_idx] == 1:
                    placements.append(
                        {
                            "item_id": item.item_id,
                            "position": self._position_coordinates(pos_idx),
                            "item": item,
                            "position_index": pos_idx,
                        }
                    )

        return placements

print("QUBO formulator defined.")

In [ ]:
class QuantumSimulator:
    """State-vector simulator for the QAOA circuit used in this example."""

    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.shots = 1024

    def apply_hadamard_layer(self, state):
        """Apply a Hadamard gate to every qubit of the state vector."""
        transformed = state.copy()
        for qubit in range(self.n_qubits):
            bit_mask = 1 << qubit
            for index in range(len(transformed)):
                if index & bit_mask == 0:
                    paired_index = index | bit_mask
                    amplitude_zero = transformed[index]
                    amplitude_one = transformed[paired_index]
                    transformed[index] = (amplitude_zero + amplitude_one) / np.sqrt(2)
                    transformed[paired_index] = (amplitude_zero - amplitude_one) / np.sqrt(2)
        return transformed

    def apply_phase_separator(self, state, gamma, Q):
        """Apply QUBO-dependent phases to computational basis states."""
        indices = np.arange(len(state))
        energies = np.zeros(len(state), dtype=float)
        bits = np.array(
            [[(index >> bit) & 1 for bit in range(self.n_qubits)] for index in indices],
            dtype=float,
        )
        energies += bits @ np.diag(Q)
        for first in range(self.n_qubits):
            for second in range(first + 1, self.n_qubits):
                energies += Q[first, second] * bits[:, first] * bits[:, second]
        return state * np.exp(-1j * gamma * energies)

    def apply_mixer_layer(self, state, beta):
        """Apply the product of single-qubit X-rotation mixers."""
        transformed = state.copy()
        cosine = np.cos(beta)
        sine = np.sin(beta)
        for qubit in range(self.n_qubits):
            bit_mask = 1 << qubit
            for index in range(len(transformed)):
                if index & bit_mask == 0:
                    paired_index = index | bit_mask
                    amplitude_zero = transformed[index]
                    amplitude_one = transformed[paired_index]
                    transformed[index] = cosine * amplitude_zero - 1j * sine * amplitude_one
                    transformed[paired_index] = cosine * amplitude_one - 1j * sine * amplitude_zero
        return transformed

    def run_qaoa_circuit(self, Q, params, p_layers):
        """Prepare the uniform state and apply p alternating QAOA layers."""
        if Q.shape != (self.n_qubits, self.n_qubits):
            raise ValueError("The QUBO dimensions must match the number of qubits.")
        if len(params) != 2 * p_layers:
            raise ValueError("QAOA requires one gamma and one beta per layer.")

        state = np.zeros(2**self.n_qubits, dtype=complex)
        state[0] = 1.0
        state = self.apply_hadamard_layer(state)

        for layer in range(p_layers):
            gamma = params[2 * layer]
            beta = params[2 * layer + 1]
            state = self.apply_phase_separator(state, gamma, Q)
            state = self.apply_mixer_layer(state, beta)

        return state

    def measure_state(self, state):
        """Sample bitstrings according to the state's probability distribution."""
        probabilities = np.abs(state) ** 2
        probabilities /= probabilities.sum()
        sampled_indices = np.random.choice(
            len(state), size=self.shots, p=probabilities
        )
        return [
            [(int(index) >> bit) & 1 for bit in range(self.n_qubits)]
            for index in sampled_indices
        ]

print("State-vector QAOA simulator defined.")

In [ ]:
class QAOAOptimizer:
    """Tune QAOA angles with a classical optimizer and sampled energies."""

    def __init__(self, qubo_matrix, p_layers=3):
        self.Q = np.asarray(qubo_matrix, dtype=float)
        if self.Q.ndim != 2 or self.Q.shape[0] != self.Q.shape[1]:
            raise ValueError("The QUBO matrix must be square.")
        self.n_qubits = self.Q.shape[0]
        self.p_layers = p_layers
        self.quantum_sim = QuantumSimulator(self.n_qubits)
        self.optimization_history = []

    def _calculate_energy(self, bitstring):
        """Evaluate the upper-triangular QUBO convention for one bitstring."""
        bits = np.asarray(bitstring, dtype=float)
        if bits.size != self.n_qubits:
            raise ValueError("The bitstring length must match the QUBO size.")
        energy = float(np.dot(np.diag(self.Q), bits))
        for first in range(self.n_qubits):
            for second in range(first + 1, self.n_qubits):
                energy += self.Q[first, second] * bits[first] * bits[second]
        return energy

    def _get_best_solution(self, measurements):
        """Return the lowest-energy measured bitstring and its energy."""
        best_solution = min(measurements, key=self._calculate_energy)
        return best_solution, self._calculate_energy(best_solution)

    def objective_function(self, params):
        """Estimate the expected QUBO energy from circuit measurements."""
        state = self.quantum_sim.run_qaoa_circuit(self.Q, params, self.p_layers)
        measurements = self.quantum_sim.measure_state(state)
        energies = [self._calculate_energy(measurement) for measurement in measurements]
        best_solution, best_energy = self._get_best_solution(measurements)
        expected_value = float(np.mean(energies))
        self.optimization_history.append(
            {
                "params": np.asarray(params).copy(),
                "expected_value": expected_value,
                "best_solution": list(best_solution),
                "best_energy": best_energy,
            }
        )
        return expected_value

    def optimize(self, max_iterations=100):
        """Run COBYLA parameter tuning and sample the final circuit state."""
        print("Starting QAOA optimization:")
        print(f"  QUBO size: {self.n_qubits} × {self.n_qubits}")
        print(f"  QAOA layers: {self.p_layers}")
        print(f"  State-vector basis states: {2**self.n_qubits}")
        print(f"  Maximum optimizer iterations: {max_iterations}")

        initial_params = np.random.uniform(0, 2 * np.pi, 2 * self.p_layers)
        result = minimize(
            self.objective_function,
            initial_params,
            method="COBYLA",
            options={"maxiter": max_iterations},
        )

        final_state = self.quantum_sim.run_qaoa_circuit(
            self.Q, result.x, self.p_layers
        )
        final_measurements = self.quantum_sim.measure_state(final_state)
        best_solution, best_energy = self._get_best_solution(final_measurements)

        return {
            "best_solution": np.asarray(best_solution, dtype=int),
            "best_energy": best_energy,
            "optimized_params": result.x,
            "optimization_success": bool(result.success),
            "optimizer_evaluations": int(result.nfev),
            "final_measurements": final_measurements,
            "convergence_history": self.optimization_history,
        }

print("QAOA optimizer defined.")

In [ ]:
def create_quantum_problem_instance():
    """Create the fixed container and eight-item teaching instance."""
    container_dims = (1.5, 1.0, 1.0)
    items = [
        QuantumContainerItem(1, 0.4, 0.3, 0.25, 3.2, 1.0),
        QuantumContainerItem(2, 0.35, 0.35, 0.3, 3.8, 1.2),
        QuantumContainerItem(3, 0.5, 0.25, 0.2, 2.9, 0.9),
        QuantumContainerItem(4, 0.3, 0.4, 0.35, 4.1, 1.1),
        QuantumContainerItem(5, 0.45, 0.3, 0.28, 3.5, 1.0),
        QuantumContainerItem(6, 0.38, 0.32, 0.3, 3.7, 0.95),
        QuantumContainerItem(7, 0.42, 0.28, 0.25, 3.1, 0.85),
        QuantumContainerItem(8, 0.33, 0.37, 0.32, 4.3, 1.15),
    ]
    return container_dims, items


container_dims, quantum_items = create_quantum_problem_instance()
container_volume = float(np.prod(container_dims))

print("Quantum problem instance:")
print(f"  Container: {container_dims[0]} m × {container_dims[1]} m × {container_dims[2]} m")
print(f"  Container volume: {container_volume:.2f} m³")
print(f"  Number of items: {len(quantum_items)}")
print(f"  Total item volume: {sum(item.volume for item in quantum_items):.2f} m³")

for item in quantum_items:
    print(
        f"  Item {item.item_id}: {item.length} m × {item.width} m × "
        f"{item.height} m; value = {item.value}"
    )

In [ ]:
print("\nFormulating the full discretized QUBO...")
qubo_formulator = QUBOFormulator(container_dims, quantum_items)
qubo_matrix = qubo_formulator.create_qubo_matrix()

print(f"  Grid resolution: {qubo_formulator.grid_resolution} m")
print(
    f"  Grid dimensions: {qubo_formulator.grid_x} × "
    f"{qubo_formulator.grid_y} × {qubo_formulator.grid_z}"
)
print(f"  Candidate origins: {qubo_formulator.total_positions}")
print(f"  Full binary-variable count: {qubo_formulator.n_variables}")
print(f"  Full QUBO matrix shape: {qubo_matrix.shape}")
print(f"  Nonzero QUBO coefficients: {np.count_nonzero(qubo_matrix)}")
print(
    f"  Full-matrix sparsity: "
    f"{100 * (1 - np.count_nonzero(qubo_matrix) / qubo_matrix.size):.1f}%"
)

# One origin per item gives eight decision variables and an eight-qubit demonstration.
# This selected subproblem does not include alternative origins for each item.
selected_variable_indices = [
    item_idx * qubo_formulator.total_positions
    for item_idx in range(len(quantum_items))
]
qubo_reduced = qubo_matrix[
    np.ix_(selected_variable_indices, selected_variable_indices)
]
n_qubits_quantum = qubo_reduced.shape[0]

print("\nSelected QAOA subproblem:")
print(f"  Binary variables / qubits: {n_qubits_quantum}")
print(f"  State-vector basis states: {2**n_qubits_quantum}")
print("  Each variable represents one item placed at the origin (0, 0, 0).")
print(f"  Reduced QUBO matrix shape: {qubo_reduced.shape}")

In [ ]:
print("\nRunning the QAOA optimization...")
qaoa_optimizer = QAOAOptimizer(qubo_reduced, p_layers=3)

start_time = time.time()
qaoa_results = qaoa_optimizer.optimize(max_iterations=50)
end_time = time.time()
optimization_time = end_time - start_time

print("\nQAOA run summary:")
print(f"  Elapsed time: {optimization_time:.2f} seconds")
print(f"  COBYLA objective evaluations: {qaoa_results['optimizer_evaluations']}")
print(f"  Optimizer reported success: {qaoa_results['optimization_success']}")
print(f"  Best measured QUBO energy: {qaoa_results['best_energy']:.3f}")
print(f"  Optimized parameters: {qaoa_results['optimized_params']}")

# Embed the eight measured decisions in the full variable vector before decoding.
binary_solution = np.zeros(qubo_formulator.n_variables, dtype=int)
binary_solution[selected_variable_indices] = qaoa_results["best_solution"]
placements = qubo_formulator.decode_solution(binary_solution)

selected_item_ids = [placement["item_id"] for placement in placements]
placed_volume = sum(placement["item"].volume for placement in placements)
volume_utilization = 100 * placed_volume / container_volume

bounds_feasible = all(
    not qubo_formulator._is_out_of_bounds(
        placement["item"], placement["position_index"]
    )
    for placement in placements
)
overlap_free = all(
    not qubo_formulator._positions_overlap(
        placements[first]["item"],
        placements[second]["item"],
        placements[first]["position_index"],
        placements[second]["position_index"],
    )
    for first in range(len(placements))
    for second in range(first + 1, len(placements))
)
assignment_unique = len(selected_item_ids) == len(set(selected_item_ids))
solution_feasible = bounds_feasible and overlap_free and assignment_unique

print("\nDecoded measured candidate:")
print(f"  Selected items: {len(placements)} of {len(quantum_items)}")
print(f"  Selected item IDs: {selected_item_ids}")
for placement in placements:
    x, y, z = placement["position"]
    print(
        f"  Item {placement['item_id']} origin: "
        f"({x:.1f}, {y:.1f}, {z:.1f}) m"
    )
print(f"  In-bounds: {bounds_feasible}")
print(f"  Overlap-free: {overlap_free}")
print(f"  At most one origin per item: {assignment_unique}")
print(f"  Feasible under modeled constraints: {solution_feasible}")
print(f"  Placed item volume: {placed_volume:.3f} m³")
print(f"  Container volume utilization: {volume_utilization:.1f}%")

assert len(binary_solution) == qubo_formulator.n_variables
assert np.all((binary_solution == 0) | (binary_solution == 1))

In [ ]:
def create_quantum_visualization():
    """Plot the selected QUBO, measured samples, and decoded placement."""
    fig = plt.figure(figsize=(16, 11))

    ax1 = fig.add_subplot(2, 2, 1)
    sns.heatmap(
        qubo_reduced,
        cmap="vlag",
        center=0,
        annot=True,
        fmt=".1f",
        cbar_kws={"label": "QUBO coefficient"},
        ax=ax1,
    )
    ax1.set_title("Eight-variable QUBO")
    ax1.set_xlabel("Decision-variable index")
    ax1.set_ylabel("Decision-variable index")

    ax2 = fig.add_subplot(2, 2, 2)
    history = qaoa_results["convergence_history"]
    if history:
        ax2.plot(
            range(1, len(history) + 1),
            [entry["expected_value"] for entry in history],
            color="navy",
            linewidth=1,
        )
        ax2.set_xlabel("Objective-function evaluation")
        ax2.set_ylabel("Sampled mean QUBO energy")
        ax2.set_title("Noisy objective estimates during COBYLA")
        ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(2, 2, 3)
    bitstring_counts = {}
    for measurement in qaoa_results["final_measurements"]:
        bitstring = "".join(str(bit) for bit in measurement)
        bitstring_counts[bitstring] = bitstring_counts.get(bitstring, 0) + 1
    most_frequent = sorted(
        bitstring_counts.items(), key=lambda entry: entry[1], reverse=True
    )[:10]
    labels = [entry[0] for entry in most_frequent]
    counts = [entry[1] for entry in most_frequent]
    ax3.bar(range(len(counts)), counts, color="slateblue")
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax3.set_xlabel("Measured bitstring (variable order)")
    ax3.set_ylabel("Number of measurements")
    ax3.set_title("Most frequent final measurements")

    ax4 = fig.add_subplot(2, 2, 4, projection="3d")
    container_x, container_y, container_z = container_dims
    corners = [
        (0, 0, 0),
        (container_x, 0, 0),
        (container_x, container_y, 0),
        (0, container_y, 0),
        (0, 0, container_z),
        (container_x, 0, container_z),
        (container_x, container_y, container_z),
        (0, container_y, container_z),
    ]
    edge_pairs = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]
    for first, second in edge_pairs:
        edge = [corners[first], corners[second]]
        ax4.plot(
            [point[0] for point in edge],
            [point[1] for point in edge],
            [point[2] for point in edge],
            color="black",
        )

    colors = plt.get_cmap("tab10")
    for placement in placements:
        item = placement["item"]
        x, y, z = placement["position"]
        dx, dy, dz = item.length, item.width, item.height
        vertices = [
            (x, y, z), (x + dx, y, z), (x + dx, y + dy, z), (x, y + dy, z),
            (x, y, z + dz), (x + dx, y, z + dz),
            (x + dx, y + dy, z + dz), (x, y + dy, z + dz),
        ]
        faces = [
            [vertices[index] for index in face]
            for face in [
                (0, 1, 2, 3), (4, 5, 6, 7), (0, 1, 5, 4),
                (2, 3, 7, 6), (0, 3, 7, 4), (1, 2, 6, 5),
            ]
        ]
        ax4.add_collection3d(
            Poly3DCollection(
                faces,
                facecolor=colors((item.item_id - 1) % 10),
                edgecolor="black",
                alpha=0.55,
            )
        )
        ax4.text(x, y, z + dz, f"Item {item.item_id}", fontsize=8)

    ax4.set_xlim(0, container_x)
    ax4.set_ylim(0, container_y)
    ax4.set_zlim(0, container_z)
    ax4.set_xlabel("Length (m)")
    ax4.set_ylabel("Width (m)")
    ax4.set_zlabel("Height (m)")
    ax4.set_title("Decoded candidate in the container")

    fig.suptitle("QAOA Container-Loading Demonstration", fontsize=15)
    fig.tight_layout()
    plt.show()


print("Creating plots from the measured run...")
create_quantum_visualization()

In [ ]:
def analyze_quantum_advantage():
    """Explain what this state-vector demonstration can and cannot show."""
    state_count = 2**qaoa_optimizer.n_qubits
    analysis = {
        "qubits": qaoa_optimizer.n_qubits,
        "basis_states": state_count,
        "full_binary_variables": qubo_formulator.n_variables,
        "sampled_candidate_feasible": solution_feasible,
        "elapsed_seconds": optimization_time,
    }

    print("\nScope of the computational demonstration")
    print("=" * 54)
    print(
        f"  State-vector qubits: {analysis['qubits']} "
        f"({analysis['basis_states']} basis states)"
    )
    print(f"  Variables in the full discretized model: {analysis['full_binary_variables']}")
    print(f"  Elapsed time for this run: {analysis['elapsed_seconds']:.2f} seconds")
    print(f"  Best measured candidate passes modeled checks: {analysis['sampled_candidate_feasible']}")
    print(
        "  The state-vector simulator stores exponentially many amplitudes "
        "as the qubit count grows."
    )
    print(
        "  This run does not compare against a classical benchmark and "
        "does not demonstrate quantum speedup."
    )
    print(
        "  QAOA is a heuristic for this demonstration; no general "
        "polynomial-time guarantee or optimality certificate is established."
    )
    return analysis


advantage_analysis = analyze_quantum_advantage()

In [ ]:
def generate_tier_9_conclusions():
    """Summarize the model, algorithm, and interpretation of this run."""
    print("QAOA container-loading demonstration: conclusions")
    print("=" * 58)
    print("\nModel and computation")
    print(f"  Full discretized binary variables: {qubo_formulator.n_variables}")
    print(f"  QAOA subproblem variables: {qaoa_optimizer.n_qubits}")
    print(f"  Alternating QAOA layers: {qaoa_optimizer.p_layers}")
    print("  Classical parameter optimizer: COBYLA")
    print(f"  COBYLA objective evaluations: {qaoa_results['optimizer_evaluations']}")
    print(f"  Best measured subproblem energy: {qaoa_results['best_energy']:.3f}")

    print("\nDecoded candidate")
    print(f"  Items selected: {len(placements)} of {len(quantum_items)}")
    print(f"  Feasible under modeled constraints: {solution_feasible}")
    print(f"  Container volume utilization: {volume_utilization:.1f}%")
    print(
        "  Volume utilization is a descriptive ratio; it is not the QUBO "
        "objective or a complete measure of loading quality."
    )

    print("\nInterpretation")
    print(
        "  The notebook illustrates QUBO construction, QAOA state evolution, "
        "classical parameter tuning, sampling, and decoding."
    )
    print(
        "  The eight-variable run considers one candidate origin per item, "
        "not every placement in the full discretized model."
    )
    print(
        "  Its state-vector simulation and measured candidate do not establish "
        "scalability, global optimality, or quantum advantage."
    )


generate_tier_9_conclusions()